In [1]:
import pandas as pd
RAW = r"D:\manager-bounce\data\raw"

games = pd.read_csv(fr"{RAW}\games.csv")
comps = pd.read_csv(fr"{RAW}\competitions.csv")

print(games.shape)
print(games.columns.tolist())
print(comps[comps['name'].str.contains('bundesliga', case=False, na=False)])

(88958, 23)
['game_id', 'competition_id', 'season', 'round', 'date', 'home_club_id', 'away_club_id', 'home_club_goals', 'away_club_goals', 'home_club_position', 'away_club_position', 'home_club_manager_name', 'away_club_manager_name', 'stadium', 'attendance', 'referee', 'url', 'home_club_formation', 'away_club_formation', 'home_club_name', 'away_club_name', 'aggregate', 'competition_type']
   competition_id competition_code        name    sub_type             type  \
0              A1       bundesliga  bundesliga  first_tier  domestic_league   
36             L1       bundesliga  bundesliga  first_tier  domestic_league   

    country_id country_name domestic_league_code confederation  total_clubs  \
0          127      Austria                   A1        europa         12.0   
36          40      Germany                   L1        europa         18.0   

                                                  url  
0   https://www.transfermarkt.co.uk/bundesliga/sta...  
36  https://www.tra

In [2]:
BL_ID = 'L1'  # replace with whatever Cell 1 showed

bl = games[games['competition_id'] == BL_ID].copy()
bl['date'] = pd.to_datetime(bl['date'])

summary = bl.groupby('season').agg(
    games=('game_id', 'nunique'),
    first=('date', 'min'),
    last=('date', 'max'),
    miss_home_mgr=('home_club_manager_name', lambda s: s.isna().sum()),
    miss_away_mgr=('away_club_manager_name', lambda s: s.isna().sum()),
)
print(summary)

        games      first       last  miss_home_mgr  miss_away_mgr
season                                                           
2012      306 2012-08-24 2013-05-18              0              0
2013      306 2013-08-09 2014-05-10              0              0
2014      306 2014-08-22 2015-05-23              0              0
2015      306 2015-08-14 2016-05-14              0              0
2016      306 2016-08-26 2017-05-20              0              0
2017      306 2017-08-18 2018-05-12              0              0
2018      306 2018-08-24 2019-05-18              0              0
2019      306 2019-08-16 2020-06-27              0              0
2020      306 2020-09-18 2021-05-22              0              0
2021      306 2021-08-13 2022-05-14              0              0
2022      306 2022-08-05 2023-05-27              0              0
2023      306 2023-08-18 2024-05-18              0              0
2024      306 2024-08-23 2025-05-17              0              0
2025      

In [3]:
cols = ['game_id', 'season', 'date']
home = bl[cols + ['home_club_id', 'home_club_manager_name']].rename(
    columns={'home_club_id': 'club_id', 'home_club_manager_name': 'manager'})
away = bl[cols + ['away_club_id', 'away_club_manager_name']].rename(
    columns={'away_club_id': 'club_id', 'away_club_manager_name': 'manager'})

long = pd.concat([home, away]).sort_values(['club_id', 'season', 'date'])
long['prev_mgr'] = long.groupby(['club_id', 'season'])['manager'].shift()

changes = long[long['manager'].notna() & long['prev_mgr'].notna()
               & (long['manager'] != long['prev_mgr'])]
print(changes.groupby('season').size())
print('Total in-season changes:', len(changes))

season
2012    10
2013    12
2014     9
2015     9
2016    12
2017    12
2018     7
2019    11
2020    14
2021    12
2022    12
2023    10
2024    12
2025    12
dtype: int64
Total in-season changes: 154


In [4]:
print(changes[changes['season'] == 2023][['date', 'club_id', 'prev_mgr', 'manager']])

            date  club_id          prev_mgr           manager
64357 2024-01-13        3  Steffen Baumgart      Timo Schultz
64293 2023-11-04       39       Bo Svensson       Jan Siewert
64404 2024-02-17       39       Jan Siewert      Bo Henriksen
64468 2024-04-13       80     Thomas Letsch    Heiko Butscher
64445 2024-03-30       82        Niko Kovac  Ralph Hasenhüttl
64308 2023-11-25       89       Urs Fischer       Marco Grote
64326 2023-12-09       89       Marco Grote     Nenad Bjelica
64497 2024-05-11       89     Nenad Bjelica       Marco Grote
64278 2023-10-22      167     Enrico Maaßen       Jess Thorup
64510 2024-05-18     2036     Frank Schmidt     Bernhard Raab
